In [ ]:
# ============================================================
# Шаг 1. Классы
# ============================================================
import json
import pickle
import time
import requests
from datetime import date
from pathlib import Path


class DailyWeather:
    """Суточные данные о погоде одного дня."""

    def __init__(self, day: date, tmax, tmin, precipitation):
        self.day = day
        self.tmax = tmax            # None, если в ответе был null
        self.tmin = tmin
        self.precipitation = precipitation

    @property
    def mean(self):
        """Среднее из максимума и минимума; None, если данных нет."""
        if self.tmax is None or self.tmin is None:
            return None
        return (self.tmax + self.tmin) / 2

    def to_dict(self):
        return {
            "date": self.day.isoformat(),           # дата строкой ISO
            "temperature_2m_max": self.tmax,
            "temperature_2m_min": self.tmin,
            "precipitation_sum": self.precipitation,
        }

    @classmethod
    def from_dict(cls, d):
        return cls(
            date.fromisoformat(d["date"]),
            d.get("temperature_2m_max"),
            d.get("temperature_2m_min"),
            d.get("precipitation_sum"),
        )

    def __eq__(self, other):
        if not isinstance(other, DailyWeather):
            return NotImplemented
        return (self.day, self.tmax, self.tmin, self.precipitation) == \
               (other.day, other.tmax, other.tmin, other.precipitation)

    def __repr__(self):
        return f"DailyWeather({self.day}, {self.tmax}, {self.tmin}, {self.precipitation})"


class CityArchive:
    """Архив погоды одного города за период."""

    def __init__(self, name, latitude, longitude, days=None):
        self.name = name
        self.latitude = latitude
        self.longitude = longitude
        self.days = days or []

    # ---------- вычисления ----------
    def _valid_means(self):
        return [d.mean for d in self.days if d.mean is not None]

    def average(self):
        means = self._valid_means()
        return sum(means) / len(means) if means else None

    def warmest_day(self):
        cand = [d for d in self.days if d.mean is not None]
        return max(cand, key=lambda d: d.mean) if cand else None

    def coldest_day(self):
        cand = [d for d in self.days if d.mean is not None]
        return min(cand, key=lambda d: d.mean) if cand else None

    def rainy_days(self, threshold=1.0):
        return sum(1 for d in self.days
                   if d.precipitation is not None and d.precipitation > threshold)

    def total_precipitation(self):
        return sum(d.precipitation for d in self.days if d.precipitation is not None)

    # ---------- сериализация ----------
    def to_dict(self):
        return {
            "city": self.name,
            "latitude": self.latitude,
            "longitude": self.longitude,
            "days": [d.to_dict() for d in self.days],
        }

    @classmethod
    def from_dict(cls, d):
        return cls(
            d["city"], d["latitude"], d["longitude"],
            [DailyWeather.from_dict(x) for x in d["days"]],
        )

    def __eq__(self, other):
        if not isinstance(other, CityArchive):
            return NotImplemented
        return (self.name, self.latitude, self.longitude, self.days) == \
               (other.name, other.latitude, other.longitude, other.days)

    def __str__(self):
        avg = self.average()
        avg_s = f"{avg:+.1f} °C" if avg is not None else "н/д"
        return (f"{self.name}: средняя {avg_s}, "
                f"осадки {self.total_precipitation():.1f} мм, "
                f"дней с осадками >1 мм: {self.rainy_days()}")


# ============================================================
# Шаг 2. Запрос с проверками и кэшем
# ============================================================
class WeatherApiError(Exception):
    """Базовая ошибка работы с Open-Meteo."""


class InvalidRequestError(WeatherApiError):
    """Ошибка некорректных параметров (HTTP 400), содержит reason сервиса."""


ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
CACHE_FILE = Path("weather_cache.json")

_api_requests = 0        # счётчик реальных запросов к API
_last_call_ts = None     # для паузы 1 секунда между запросами

CITIES = {
    "Москва":          (55.7558, 37.6173),
    "Санкт-Петербург": (59.9386, 30.3141),
    "Сочи":            (43.5855, 39.7231),
    "Якутск":          (62.0355, 129.6755),
    "Калининград":     (54.7104, 20.4522),
}
DAILY_PARAMS = "temperature_2m_max,temperature_2m_min,precipitation_sum"


def _pause():
    """Гарантирует паузу не меньше 1 секунды между запросами."""
    global _last_call_ts
    if _last_call_ts is not None:
        elapsed = time.monotonic() - _last_call_ts
        if elapsed < 1.0:
            time.sleep(1.0 - elapsed)
    _last_call_ts = time.monotonic()


def _load_cache():
    if CACHE_FILE.exists():
        with open(CACHE_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def _parse_city(name, lat, lon, payload):
    daily = payload.get("daily") or {}
    times = daily.get("time") or []

    def get_val(key, i):
        arr = daily.get(key) or []
        return arr[i] if i < len(arr) else None

    days = []
    for i, day_str in enumerate(times):
        days.append(DailyWeather(
            date.fromisoformat(day_str),
            get_val("temperature_2m_max", i),      # null -> None
            get_val("temperature_2m_min", i),
            get_val("precipitation_sum", i),
        ))
    return CityArchive(name, lat, lon, days)


def fetch_city(name, lat, lon,
               start_date="2026-03-01", end_date="2026-03-31",
               url=ARCHIVE_URL):
    """Возвращает CityArchive; кэширует ответы, считает реальные запросы."""
    global _api_requests

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": DAILY_PARAMS,
        "timezone": "Europe/Moscow",
    }
    key = f"{name}|{start_date}|{end_date}"     # ключ: город + период

    cache = _load_cache()
    if key in cache:                            # архивные данные не меняются
        return _parse_city(name, lat, lon, cache[key])

    _pause()
    _api_requests += 1
    try:
        resp = requests.get(url, params=params, timeout=15)
    except requests.RequestException as e:
        raise WeatherApiError(f"Сетевая ошибка при запросе «{name}»: {e}") from e

    if resp.status_code == 400:
        try:
            reason = resp.json().get("reason", resp.text)
        except ValueError:
            reason = resp.text
        raise InvalidRequestError(reason)

    if resp.status_code != 200:
        raise WeatherApiError(f"HTTP {resp.status_code}: {resp.text[:200]}")

    payload = resp.json()
    cache[key] = payload
    _save_cache(cache)
    return _parse_city(name, lat, lon, payload)


# ============================================================
# Шаг 3–4. Основной сценарий
# ============================================================
def fmt_day(d):
    """'28.03 (+16.9)' или прочерк."""
    return f"{d.day.strftime('%d.%m')} ({d.mean:+.1f})" if d else "—"


def print_summary(archives):
    header = (f"{'Город':<18}{'Средняя':>9}  "
              f"{'Самый тёплый':<16}{'Самый холодный':<16}"
              f"{'Осадки':>8}{'Дней >1 мм':>12}")
    print(header)
    for a in archives:                          # уже отсортированы по средней
        avg = a.average()
        print(f"{a.name:<18}{avg:>+9.1f}  "
              f"{fmt_day(a.warmest_day()):<16}"
              f"{fmt_day(a.coldest_day()):<16}"
              f"{a.total_precipitation():>8.1f}"
              f"{a.rainy_days():>12}")


def main():
    # ---- 1. Получение данных за март 2026 ----
    archives = [fetch_city(name, lat, lon, "2026-03-01", "2026-03-31")
                for name, (lat, lon) in CITIES.items()]

    print(f"Запросов к API: {_api_requests} "
          f"(повторный запуск: 0 — данные из кэша)")

    # ---- 2. Сохранение и проверка обратимости ----
    with open("weather.json", "w", encoding="utf-8") as f:
        json.dump([a.to_dict() for a in archives], f,
                  ensure_ascii=False, indent=2)
    with open("weather.pkl", "wb") as f:
        pickle.dump(archives, f)

    with open("weather.json", encoding="utf-8") as f:
        loaded_json = [CityArchive.from_dict(d) for d in json.load(f)]
    with open("weather.pkl", "rb") as f:
        loaded_pkl = pickle.load(f)

    json_size = Path("weather.json").stat().st_size
    pkl_size = Path("weather.pkl").stat().st_size

    # ---- 3. Сводка ----
    print()
    ranked = sorted(archives,
                    key=lambda a: a.average() if a.average() is not None
                    else float("-inf"),
                    reverse=True)               # от тёплого к холодному
    print_summary(ranked)

    warmest = ranked[0]
    coldest = ranked[-1]
    wettest = max(archives, key=lambda a: a.total_precipitation())

    print(f"\nСамый тёплый город:   {warmest.name} ({warmest.average():+.1f} °C)")
    print(f"Самый холодный город: {coldest.name} ({coldest.average():+.1f} °C)")
    print(f"Больше всего осадков: {wettest.name} ({wettest.total_precipitation():.1f} мм)")
    print(f"Разница самый тёплый — самый холодный: "
          f"{warmest.average() - coldest.average():.1f} °C")

    print(f"\nJSON: {json_size} байт, pickle: {pkl_size} байт; "
          f"загрузка без потерь: {loaded_json == archives} / {loaded_pkl == archives}")

    # ---- 4. Проверка обработки ошибок ----
    print("\n--- Проверка обработки ошибок ---")

    # 4.1 широта 120
    try:
        fetch_city("Тест-широта", 120.0, 37.6173, "2026-03-01", "2026-03-31")
    except InvalidRequestError as e:
        print(f"Ошибка запроса: {e}")
    except WeatherApiError as e:
        print(f"Ошибка API: {e}")

    # 4.2 период в будущем
    try:
        fetch_city("Москва-будущее", 55.7558, 37.6173,
                   "2027-01-01", "2027-01-31")
    except InvalidRequestError as e:
        print(f"Ошибка запроса: {e}")
    except WeatherApiError as e:
        print(f"Ошибка API: {e}")

    # 4.3 несуществующий адрес
    try:
        fetch_city("Несуществующий", 55.0, 37.0, "2026-03-01", "2026-03-31",
                   url="https://archive-api.open-meteo.com/v1/nothing")
    except InvalidRequestError as e:
        print(f"Ошибка запроса: {e}")
    except WeatherApiError as e:
        print(f"Ошибка API: {e}")

    # ---- 5. Дополнительно: март 2025 ----
    print("\n--- Сравнение с мартом 2025 ---")
    archives_2025 = {name: fetch_city(name, lat, lon, "2025-03-01", "2025-03-31")
                     for name, (lat, lon) in CITIES.items()}

    deltas = {}
    for a in archives:
        old = archives_2025[a.name].average()
        new = a.average()
        if old is not None and new is not None:
            deltas[a.name] = new - old

    for name, d in sorted(deltas.items(), key=lambda kv: kv[1], reverse=True):
        print(f"  {name:<18}{d:+.1f} °C")

    if deltas:
        leader = max(deltas, key=lambda n: abs(deltas[n]))
        print(f"Сильнее всего средняя мартовская температура изменилась в: "
              f"{leader} ({deltas[leader]:+.1f} °C)")

    print(f"\nВсего запросов к API за запуск (включая 2025 и проверки ошибок): "
          f"{_api_requests}")


if __name__ == "__main__":
    main()